# Ekstraksi Fitur Warna dan Tekstur Citra Wajah

## Import Library & Mount Drive

In [25]:
import cv2
import numpy as np
import pandas as pd
import os

from google.colab import drive
drive.mount('/content/drive')

from skimage.feature import graycomatrix, graycoprops

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Tentukan Path Dataset

In [26]:
base_path = "/content/drive/MyDrive/datasets/citra-male-female"

male_path = os.path.join(base_path, "male")
female_path = os.path.join(base_path, "female")

## Fungsi Ekstraksi Fitur HSV

In [27]:
def extract_hsv(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    h_mean = np.mean(hsv[:,:,0])
    s_mean = np.mean(hsv[:,:,1])
    v_mean = np.mean(hsv[:,:,2])

    return [h_mean, s_mean, v_mean]

## Fungsi Ekstraksi Fitur Tekstur (GLCM)

In [28]:
def extract_glcm(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)

    contrast = graycoprops(glcm, 'contrast')[0,0]
    correlation = graycoprops(glcm, 'correlation')[0,0]
    energy = graycoprops(glcm, 'energy')[0,0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0,0]

    return [contrast, correlation, energy, homogeneity]

## Loop Semua Gambar + Labeling

In [29]:
data = []
labels = []

def process_folder(folder_path, label):
    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)

        image = cv2.imread(img_path)

        if image is None:
            continue

        image = cv2.resize(image, (58, 58))  # samakan ukuran

        hsv_feat = extract_hsv(image)
        glcm_feat = extract_glcm(image)

        features = hsv_feat + glcm_feat

        data.append(features)
        labels.append(label)

Jalankan untuk Male & Female

In [30]:
process_folder(male_path, "male")
process_folder(female_path, "female")

## Gabungkan ke DataFrame

In [31]:
columns = [
    "h_mean", "s_mean", "v_mean",
    "contrast", "correlation", "energy", "homogeneity"
]

df = pd.DataFrame(data, columns=columns)
df["label"] = labels

print("3 data pertama:")
display(df.head(3))
print("\n3 data terakhir:")
display(df.tail(3))

3 data pertama:


,h_mean,s_mean,v_mean,contrast,correlation,energy,homogeneity,label
0,146.635256,43.142093,180.267836,252.042952,0.969749,0.021202,0.149806,male
1,14.199762,107.114447,185.789834,450.358137,0.894740,0.022071,0.134727,male
2,25.746136,79.696492,160.725030,319.527526,0.931651,0.018015,0.119173,male



3 data terakhir:


,h_mean,s_mean,v_mean,contrast,correlation,energy,homogeneity,label
17,35.263971,143.475922,97.900713,139.768300,0.977749,0.036823,0.222148,female
18,18.158145,99.627824,147.504756,420.329099,0.932228,0.024665,0.180526,female
19,19.564804,90.778240,128.579667,235.032970,0.964639,0.022661,0.137120,female


## Export ke CSV

In [32]:
output_path = "/content/drive/MyDrive/datasets/fitur_wajah.csv"
df.to_csv(output_path, index=False)

print("Berhasil disimpan di:", output_path)

Berhasil disimpan di: /content/drive/MyDrive/datasets/fitur_wajah.csv


# Klasifikasi Gender Berdasarkan Citra Wajah
*tambahan

## Import & Persiapan Data

In [33]:
import cv2, os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load dataset hasil ekstraksi
df = pd.read_csv("/content/drive/MyDrive/datasets/fitur_wajah.csv")

X = df.drop("label", axis=1)
y = df["label"]

## Training Model

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

# Evaluasi
y_pred = model.predict(X_test)
print("Akurasi:", accuracy_score(y_test, y_pred))

Akurasi: 0.25


## Testing (Prediksi Gambar Baru)

In [35]:
test_path = "/content/drive/MyDrive/datasets/citra-male-female/test"
results = []

for file in sorted(os.listdir(test_path)):
    img = cv2.imread(os.path.join(test_path, file))
    if img is None: continue

    img = cv2.resize(img, (58,58))

    hsv = extract_hsv(img)
    glcm = extract_glcm(img)

    fitur = np.array(hsv + glcm).reshape(1, -1)
    pred = model.predict(fitur)[0]

    results.append([file, pred])

df_result = pd.DataFrame(results, columns=["nama_file","prediksi"])
df_result

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


,nama_file,prediksi
0,1.jpg,female
1,2.jpg,male
2,3.jpg,male
3,4.jpg,female
